In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mrmr import mrmr_classif
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.metrics import confusion_matrix, roc_curve, auc
import time
import os
import csv
import shap

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
class MultiHeadSelfAttentionClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=4, num_heads=8, hidden_dim=512):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=input_dim, num_heads=num_heads)
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x.unsqueeze(0)
        attn_output, _ = self.attention(x, x, x)
        x = attn_output.squeeze(0)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
class ComplexLinearClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=4):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)  # First hidden layer
        self.fc2 = nn.Linear(128, 64)  # Second hidden layer
        self.fc3 = nn.Linear(64, num_classes)  # Output layer
        self.dropout = nn.Dropout(0.5)  # Dropout for regularization

    def forward(self, x):
        x = F.relu(self.fc1(x))  # Activation after first layer
        x = self.dropout(x)  # Apply dropout
        x = F.relu(self.fc2(x))  # Activation after second layer
        x = self.fc3(x)  # Output layer (no activation, handled by loss function)
        return x

In [ ]:
def calculate_metrics(y_true, y_pred, y_pred_proba, n_classes):
    acc = accuracy_score(y_true, y_pred)
    auc_score = roc_auc_score(y_true, y_pred_proba, multi_class='ovr')
    pre = precision_score(y_true, y_pred, average='macro')
    rec = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')
    mcc = matthews_corrcoef(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)
    specificity = []
    sensitivity = []
    for i in range(n_classes):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        fn = cm[i, :].sum() - cm[i, i]
        tp = cm[i, i]
        specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
        sensitivity.append(tp / (tp + fn) if (tp + fn) > 0 else 0)

    sp_avg = np.mean(specificity)
    sn_avg = np.mean(sensitivity)

    return {
        'accuracy': acc,
        'auc': auc_score,
        'precision': pre,
        'recall': rec,
        'specificity': sp_avg,
        'sensitivity': sn_avg,
        'f1': f1,
        'mcc': mcc
    }

In [ ]:
def plot_roc_curves(y_test, y_pred_probas, class_names, method_names, n_classes):
    plt.figure(figsize=(12, 10))

    colors = ['blue', 'green', 'red', 'purple', 'orange']  # Added one more color for SHAP
    linestyles = ['-', '--', '-.', ':', '-']  # Added one more linestyle for SHAP

    for m_idx, (method_name, y_pred_proba) in enumerate(zip(method_names, y_pred_probas)):
        # Calculate ROC curve and ROC area for each class
        fpr = dict()
        tpr = dict()
        roc_auc = dict()

        for i in range(n_classes):
            # For each class, create a binary classification problem
            y_true_binary = (y_test == i).astype(int)
            y_score = y_pred_proba[:, i]

            fpr[i], tpr[i], _ = roc_curve(y_true_binary, y_score)
            roc_auc[i] = auc(fpr[i], tpr[i])

            # Plot ROC curve for each class
            plt.plot(
                fpr[i],
                tpr[i],
                color=colors[m_idx],
                linestyle=linestyles[i],
                lw=3,
                label=f'{method_name} - {class_names[i]} (AUC = {roc_auc[i]:.2f})'
            )

    # Plot the diagonal line
    # plt.plot([0, 1], [0, 1], 'k--', lw=2)

    # Set plot formatting
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=24)
    plt.ylabel('True Positive Rate', fontsize=24)
    plt.title('Receiver Operating Characteristic (ROC) Curves', fontsize=28)
    plt.legend(loc="lower right", fontsize=18)

    # Set tick parameters
    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)

    # Save the plot
    plt.tight_layout()
    plt.savefig('roc_curves_comparison.png', dpi=1000)
    plt.savefig('roc_curves_comparison.pdf', format='pdf', dpi=1000)
    plt.close()

In [ ]:
# def run_shap_analysis(X, y, class_names):
#     print("\nRunning simplified SHAP analysis...")
# 
#     # Input dimension and classes
#     input_dim = X.shape[1]
#     n_classes = len(class_names)
# 
#     # Split data
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 
#     # Simplified model with fewer parameters
#     model = nn.Linear(input_dim, n_classes)  # Simple linear model instead of ComplexLinearClassifier
# 
#     # Convert to tensors
#     X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
#     X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
#     y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
# 
#     # Faster training with fewer epochs
#     criterion = nn.CrossEntropyLoss()
#     optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Increased learning rate
#     num_epochs = 5  # Reduced epochs
#     
#     model.train()
#     for epoch in range(num_epochs):
#         optimizer.zero_grad()
#         outputs = model(X_train_tensor)
#         loss = criterion(outputs, y_train_tensor)
#         loss.backward()
#         optimizer.step()
#         print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item():.4f}")
# 
#     # Quick evaluation
#     model.eval()
#     with torch.no_grad():
#         outputs = model(X_test_tensor)
#         y_pred = torch.argmax(outputs, dim=1)
#         y_pred_proba = torch.softmax(outputs, dim=1)
# 
#     # Convert to numpy
#     y_test_np = y_test_tensor.numpy()
#     y_pred_np = y_pred.numpy()
#     y_pred_proba_np = y_pred_proba.numpy()
# 
#     # Basic metrics
#     metrics = {
#         'accuracy': accuracy_score(y_test_np, y_pred_np),
#         'method': 'Simplified SHAP',
#         'num_features': input_dim
#     }
# 
#     # Simplified SHAP analysis with smaller samples
#     def f(x):
#         model.eval()
#         with torch.no_grad():
#             x_tensor = torch.tensor(x, dtype=torch.float32)
#             return torch.softmax(model(x_tensor), dim=1).numpy()
# 
#     # Use smaller subsets
#     background = X_train.values[:50]  # Reduced from 100
#     shap_input = X_test.values[:50]   # Reduced from 200
# 
#     # Single SHAP explainer for all classes
#     explainer = shap.KernelExplainer(f, background)
#     shap_values = explainer.shap_values(shap_input)
# 
#     # Generate one summary plot for all classes
#     plt.figure(figsize=(10, 6))
#     shap.summary_plot(shap_values, shap_input, feature_names=X.columns, 
#                      class_names=class_names, show=False, max_display=10)
#     plt.title("SHAP Summary Plot", fontsize=14)
#     plt.tight_layout()
#     plt.savefig('shap_summary.png', dpi=300)  # Reduced DPI for faster saving
#     plt.close()
# 
#     return metrics, y_test_np, y_pred_proba_np

In [ ]:
def run_experiment(X, y, class_names, feature_selection_method, n_features=1000, num_epochs=50):
    n_classes = len(class_names)

    # Feature selection
    print(f"\nRunning feature selection: {feature_selection_method}")
    if feature_selection_method == 'MRMR':
        selected_features = mrmr_classif(X=X, y=y, K=n_features)
        X_selected = X[selected_features]
        input_dim = len(selected_features)
        feature_names = selected_features

    elif feature_selection_method == 'LASSO':
        lasso = LassoCV(cv=5, random_state=42).fit(X, y)
        selected_features = X.columns[np.abs(lasso.coef_) > 1e-5]
        X_selected = X[selected_features]
        input_dim = len(selected_features)
        feature_names = selected_features

    elif feature_selection_method == 'PCA':
        pca = PCA(n_components=n_features, random_state=42)
        X_selected = pca.fit_transform(X)
        input_dim = X_selected.shape[1]
        # For PCA, we don't have meaningful feature names
        feature_names = [f"PC_{i}" for i in range(input_dim)]
        # Convert to DataFrame for consistency
        X_selected = pd.DataFrame(X_selected, columns=feature_names)

    elif feature_selection_method == 'CHI2':
        # Ensure non-negative values for Chi-Squared
        X_shifted = X - X.min().min() if X.min().min() < 0 else X
        selector = SelectKBest(score_func=chi2, k=n_features)
        X_selected = selector.fit_transform(X_shifted, y)
        # Get selected feature indices
        selected_indices = selector.get_support(indices=True)
        feature_names = X.columns[selected_indices].tolist()
        # Convert to DataFrame for consistency
        X_selected = pd.DataFrame(X_selected, columns=feature_names)
        input_dim = X_selected.shape[1]

    # Print the number of features selected
    print(f"Number of features selected: {input_dim}")

    # Ensure num_heads is a divisor of input_dim
    num_heads = min(8, input_dim)
    while input_dim % num_heads != 0:
        num_heads -= 1

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

    # Convert to PyTorch tensors
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    if isinstance(X_train, pd.DataFrame):
        X_train_tensor = torch.FloatTensor(X_train.values).to(device)
        X_test_tensor = torch.FloatTensor(X_test.values).to(device)
    else:
        X_train_tensor = torch.FloatTensor(X_train).to(device)
        X_test_tensor = torch.FloatTensor(X_test).to(device)

    y_train_tensor = torch.LongTensor(y_train.values).to(device)
    y_test_tensor = torch.LongTensor(y_test.values).to(device)

    # Initialize model
    model = MultiHeadSelfAttentionClassifier(
        input_dim=input_dim,
        num_classes=n_classes,
        num_heads=num_heads
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # Training
    start_time = time.time()
    model.train()
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item():.4f}")

    training_time = time.time() - start_time

    # Testing
    start_time = time.time()
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        y_pred = torch.argmax(outputs, dim=1)
        y_pred_proba = torch.softmax(outputs, dim=1)
    testing_time = time.time() - start_time

    # Convert to numpy
    y_test_np = y_test_tensor.cpu().numpy()
    y_pred_np = y_pred.cpu().numpy()
    y_pred_proba_np = y_pred_proba.cpu().numpy()

    # Calculate metrics
    metrics = calculate_metrics(y_test_np, y_pred_np, y_pred_proba_np, n_classes)
    metrics['training_time'] = training_time
    metrics['testing_time'] = testing_time
    metrics['method'] = feature_selection_method
    metrics['num_features'] = input_dim

    # Save feature importance information
    if feature_selection_method in ['MRMR', 'LASSO', 'CHI2']:
        feature_importance_df = pd.DataFrame({
            'feature': feature_names,
            'method': feature_selection_method
        })
        feature_importance_df.to_csv(f'{feature_selection_method}_selected_features.csv', index=False)
        print(
            f"Selected features for {feature_selection_method} saved to {feature_selection_method}_selected_features.csv")

    return metrics, y_test_np, y_pred_proba_np

In [ ]:
def plot_roc_curves_by_class(y_test, y_pred_probas, class_names, method_names, n_classes):
    colors = ['blue', 'green', 'red', 'purple', 'orange']  # Added one more color for SHAP

    # Create a separate plot for each class
    for i in range(n_classes):
        plt.figure(figsize=(6, 6))
        plt.rcParams['font.family'] = 'Times New Roman'

        for m_idx, (method_name, y_pred_proba) in enumerate(zip(method_names, y_pred_probas)):
            # For each class, create a binary classification problem
            y_true_binary = (y_test == i).astype(int)
            y_score = y_pred_proba[:, i]

            fpr, tpr, _ = roc_curve(y_true_binary, y_score)
            roc_auc = auc(fpr, tpr)

            # Plot ROC curve for this class and method
            plt.plot(
                fpr,
                tpr,
                color=colors[m_idx],
                lw=3,
                label=f'{method_name} (AUC = {roc_auc:.2f})'
            )

        # Plot the diagonal line
        plt.plot([0, 1], [0, 1], 'k--', lw=2)

        # Set plot formatting
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])

        for spine in plt.gca().spines.values():
            spine.set_linewidth(2)
            spine.set_edgecolor('black')

        plt.xlabel('False Positive Rate', fontsize=24)
        plt.ylabel('True Positive Rate', fontsize=24)
        plt.title(f'ROC Curve for Class: {class_names[i]}', fontsize=28, pad=20)
        plt.legend(loc="lower right", fontsize=18)

        # Set tick parameters
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)

        # Save the plot
        plt.tight_layout()
        plt.savefig(f'roc_curve_class_{i}_{class_names[i]}.png', dpi=1000)
        plt.savefig(f'roc_curve_class_{i}_{class_names[i]}.pdf', format='pdf', dpi=1000)
        plt.close()

In [ ]:
def plot_feature_importance_comparison(feature_importance_data, top_n=20):
    # Assumes feature_importance_data is a list of tuples: [(method_name, features, importance_values)]
    plt.figure(figsize=(15, 10))

    bar_width = 0.2
    colors = ['blue', 'green', 'red', 'purple', 'orange']

    for idx, (method, features, importances) in enumerate(feature_importance_data):
        # Sort features by importance
        sorted_indices = np.argsort(importances)[-top_n:]
        sorted_features = [features[i] for i in sorted_indices]
        sorted_importances = [importances[i] for i in sorted_indices]

        # Plot bars
        x = np.arange(len(sorted_features))
        plt.barh(x + idx * bar_width, sorted_importances, bar_width,
                 color=colors[idx], label=method)

    plt.yticks(np.arange(top_n) + bar_width, sorted_features, fontsize=12)
    plt.xlabel('Importance', fontsize=14)
    plt.title(f'Top {top_n} Features by Importance', fontsize=16)
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(f'feature_importance_comparison_top{top_n}.png', dpi=1000)
    plt.savefig(f'feature_importance_comparison_top{top_n}.pdf', format='pdf', dpi=1000)
    plt.close()

In [ ]:
# Optimized SHAP analysis
def run_shap_analysis_fast(X, y, class_names):
    print("\nRunning fast SHAP analysis...")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = nn.Linear(X.shape[1], len(class_names))
    
    # Quick training
    X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    model.train()
    for epoch in range(5):  # Minimal epochs
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()

    # Quick evaluation
    X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_tensor)
        y_pred = torch.argmax(outputs, dim=1)
        y_pred_proba = torch.softmax(outputs, dim=1)

    y_test_np = y_test.values
    y_pred_np = y_pred.numpy()
    y_pred_proba_np = y_pred_proba.numpy()

    # Calculate confusion matrix for SP and SN
    cm = confusion_matrix(y_test_np, y_pred_np)
    if len(class_names) == 2:  # Binary classification
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    else:  # Multi-class: average per-class metrics
        specificity = []
        sensitivity = []
        for i in range(len(class_names)):
            tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
            fp = cm[:, i].sum() - cm[i, i]
            fn = cm[i, :].sum() - cm[i, i]
            tp = cm[i, i]
            sp = tn / (tn + fp) if (tn + fp) > 0 else 0
            sn = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity.append(sp)
            sensitivity.append(sn)
        specificity = np.mean(specificity)
        sensitivity = np.mean(sensitivity)

    # Calculate all metrics
    metrics = {
        'accuracy': accuracy_score(y_test_np, y_pred_np),
        'auc': roc_auc_score(y_test_np, y_pred_proba_np, multi_class='ovr'),
        'precision': precision_score(y_test_np, y_pred_np, average='macro', zero_division=0),
        'specificity': specificity,
        'sensitivity': sensitivity,
        'f1': f1_score(y_test_np, y_pred_np, average='macro', zero_division=0),
        'mcc': matthews_corrcoef(y_test_np, y_pred_np),
        'method': 'SHAP'
    }

    return metrics, y_test_np, y_pred_proba_np

In [ ]:
# Load data
print("Loading data...")
data = pd.read_csv('../3_Feature_Normalization/Hybrid_ConvNeXt_NASNetLarge/files/standardized_data_robust.csv')
X = data.iloc[:, :-3]  # Features
y = data['label']  # Labels

# Get unique class names mapped to labels
class_mapping = data[['label', 'class']].drop_duplicates().sort_values('label')
class_names = class_mapping['class'].tolist()
n_classes = len(class_names)

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {n_classes}")
print(f"Class names: {class_names}")

# Feature selection methods to compare
feature_selection_methods = ['MRMR', 'LASSO', 'PCA', 'CHI2']

# Adjust the number of features for each method
n_features = {
    'MRMR': 1000,
    'LASSO': 500,  # LASSO will select its own features based on regularization
    'PCA': 50,
    'CHI2': 200
}

# Store results
all_results = []
y_test_data = None
y_pred_probas = []
method_names = []

# Run experiments for each feature selection method
for method in feature_selection_methods:
    print(f"\n{'-' * 50}")
    print(f"Running experiment with {method} feature selection")
    print(f"{'-' * 50}")

    metrics, y_test, y_pred_proba = run_experiment(
        X, y, class_names,
        feature_selection_method=method,
        n_features=n_features[method],
        num_epochs=50
    )

    all_results.append(metrics)

    # Store data for ROC curve
    if y_test_data is None:
        y_test_data = y_test
    y_pred_probas.append(y_pred_proba)
    method_names.append(method)

    # Print results
    print(f"\nResults for {method}:")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"AUC: {metrics['auc']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"MCC: {metrics['mcc']:.4f}")
    print(f"Training Time: {metrics['training_time']:.4f} s")
    print(f"Testing Time: {metrics['testing_time']:.4f} s")

# Now run SHAP analysis
shap_metrics, shap_y_test, shap_y_pred_proba = run_shap_analysis_fast(X, y, class_names)
all_results.append(shap_metrics)

# Add SHAP results to ROC curve data
y_pred_probas.append(shap_y_pred_proba)
method_names.append('SHAP')

# Print SHAP results
print(f"\nResults for SHAP:")
print(f"Accuracy: {shap_metrics['accuracy']:.4f}")
print(f"AUC: {shap_metrics['auc']:.4f}")
print(f"F1 Score: {shap_metrics['f1']:.4f}")
print(f"MCC: {shap_metrics['mcc']:.4f}")
print(f"Sensitivity: {shap_metrics['sensitivity']:.4f}")
print(f"Specificity: {shap_metrics['specificity']:.4f}")

# Export results to CSV
results_df = pd.DataFrame(all_results)
results_df.to_csv('feature_selection_comparison_results.csv', index=False)
print("\nResults saved to feature_selection_comparison_results.csv")

# Plot separate ROC curves for each class
print("Plotting separate ROC curves for each class...")
plot_roc_curves_by_class(y_test_data, y_pred_probas, class_names, method_names, n_classes)
print(f"ROC curves saved as individual files for each of the {n_classes} classes")

# Combined ROC curves
plot_roc_curves(y_test_data, y_pred_probas, class_names, method_names, n_classes)
print("Combined ROC curves saved to roc_curves_comparison.png and roc_curves_comparison.pdf")

# Print comparison
print("\nComparison of all methods (including SHAP):")
for metric in ['accuracy', 'auc', 'f1', 'mcc']:
    print(f"\n{metric.upper()}:")
    for method, result in zip(method_names, all_results):
        print(f"{method}: {result[metric]:.4f}")

# Create a bar chart comparing all methods
metrics_to_plot = ['accuracy', 'auc', 'f1', 'mcc', 'specificity', 'sensitivity']
plt.figure(figsize=(14, 10))

x = np.arange(len(metrics_to_plot))
width = 0.15
multiplier = 0

for method, result in zip(method_names, all_results):
    offset = width * multiplier
    values = [result[metric] for metric in metrics_to_plot]
    plt.bar(x + offset, values, width, label=method)
    multiplier += 1

plt.xlabel('Metrics', fontsize=18)
plt.ylabel('Values', fontsize=18)
plt.title('Performance Metrics Comparison Across Methods', fontsize=24)
plt.xticks(x + width * (len(method_names) - 1) / 2, metrics_to_plot, fontsize=16)
plt.yticks(fontsize=16)
plt.legend(loc='lower right', fontsize=16)
plt.tight_layout()

plt.savefig('metrics_comparison.png', dpi=1000)
plt.savefig('metrics_comparison.pdf', format='pdf', dpi=1000)
plt.close()
print("Metrics comparison chart saved as metrics_comparison.png and metrics_comparison.pdf")